# Create boundary from a Steve mosaic

Derives `boundary_positions*.txt`/`hole*.txt` automatically from a low-mag
Steve mosaic (e.g. `data/mosaic10x/`), instead of drawing them by hand.
Writes into `positions/boundaries/from_mosaic/`, in the exact convention
`02_create_positions_from_boundaries.ipynb` already expects (that notebook's
`BOUNDARY_SOURCE` picks between this and `positions/boundaries/manual/`, if
you also have hand-drawn boundaries) -- run this notebook first, then that
one.

**Method:** the mosaic's tiles (each already stage-position-tagged by Steve)
are pasted into one flattened image; the image is smoothed, thresholded, and
cleaned up morphologically (closing to bridge small real gaps, opening to
drop noise specks, then dilated outward by a small margin) to get a tissue
mask; enclosed background regions inside the mask become holes -- including
a hole that is really a donut/annulus around a real tissue island, which
becomes an interior ring of the hole polygon rather than being swallowed
whole. Both tissue and holes are traced into polygons in real stage-micron
coordinates -- see `MERci.acquisition.mosaic` for the full pipeline.

**Mixed-objective tiles.** A mosaic can include tiles shot at a different
objective than the main scan (e.g. a handful of 60x alignment/reference
FOVs, deliberately overlapping the 10x scan). In practice compositing those
in only hurts the low-mag tissue segmentation (different exposure/gain adds
a spurious mode to the intensity histogram and confuses thresholding), so by
default only the tiles matching `MOSAIC_KEEP_OBJECTIVES` below are used.
Widen it (e.g. add `"60x"`) only if some sample genuinely needs those tiles
composited in. Tiles sharing the *same* kept objective (e.g. several 10x
tiles) can still legitimately overlap each other (re-scanned regions) --
that's handled directly by `assemble_mosaic_canvas`'s own stacking-order
compositing, no extra parameter needed.

**Picking a threshold.** Otsu (the automatic default in `segment_mosaic_tissue`)
doesn't work well on every sample. The histogram cell below overlays every
tile's log-space intensity histogram plus the pooled combined histogram, and
when that combined histogram is clearly bimodal, automatically estimates the
valley between the background and tissue-signal peaks -- that estimate seeds
`THRESHOLD` for the first segmentation attempt, rather than starting from
Otsu.

**This is a visual, iterative notebook, not a one-shot script.** Re-run the
segmentation + review cells with different threshold/morphology parameters
until the green (tissue) / red (hole) overlay in the review plot looks right,
*then* run the final write cell -- it does not run automatically.

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity
from MERci.acquisition.mosaic import (
    load_steve_mosaic, assemble_mosaic_canvas,
    plot_tile_intensity_histograms, segment_mosaic_tissue,
    plot_mosaic_segmentation, save_boundary_from_mosaic,
)

POSITIONS_DIR = SAMPLE_DIR / "positions"
# This notebook's output source, distinct from positions/boundaries/manual/
# (hand-drawn) -- 02_create_positions_from_boundaries.ipynb's BOUNDARY_SOURCE
# picks between the two.
OUTPUT_DIR = POSITIONS_DIR / "boundaries" / "from_mosaic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_NAME is the TRUE top-level experiment id -- see notebook 02's own
# docstring (02_create_positions_from_boundaries.ipynb) for why this isn't
# SAMPLE_DIR.name. Not used for naming here (this notebook only writes
# boundary_positions*.txt/hole*.txt, which carry no sample name), but printed
# for confirmation that MERCI_DIR resolves to the experiment you expect.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")
print(f"SAMPLE_NAME: {SAMPLE_NAME}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

## Load the mosaic

`MOSAIC_DIR` defaults to `SAMPLE_DIR/data/mosaic10x` -- the folder
`02_create_positions_from_boundaries.ipynb` already creates as a placeholder,
and where Steve's own "Save Mosaic" writes its `.msc` manifest + `.stv` tile
files. Set `MOSAIC_DIR`/`MOSAIC_NAME` manually if yours lives elsewhere or is
named differently.

In [ ]:
MOSAIC_DIR  = SAMPLE_DIR / "data" / "mosaic10x"
MOSAIC_NAME = None   # e.g. "mosaic10x" -- None = auto-detect the only *.msc file present

msc_candidates = sorted(MOSAIC_DIR.glob(f"{MOSAIC_NAME or '*'}.msc"))
if not msc_candidates:
    raise FileNotFoundError(
        f"No .msc mosaic manifest found in {MOSAIC_DIR} -- copy Steve's saved "
        f"mosaic there first (or set MOSAIC_DIR/MOSAIC_NAME above)."
    )
if len(msc_candidates) > 1:
    print(f"WARNING: {len(msc_candidates)} .msc files found, using the first: "
          f"{msc_candidates[0].name}. Set MOSAIC_NAME to pick a different one.")
MSC_PATH = msc_candidates[0]

tiles_all = load_steve_mosaic(MSC_PATH)
print(f"Loaded {len(tiles_all)} tile(s) from {MSC_PATH.name}")

obj_counts = Counter(t.objective_name for t in tiles_all)
print(f"Objective breakdown: {dict(obj_counts)}")

# Only tiles whose objective is in this list are composited into the canvas
# below -- a handful of higher-mag alignment/reference FOVs (e.g. "60x")
# deliberately overlapping the low-mag scan were found to only hurt tissue
# segmentation (their different exposure/gain adds a spurious mode to the
# intensity histogram and confuses thresholding), not help it. Set to None
# to keep every objective (the old default), or list more than one objective
# to composite several. Tiles sharing a kept objective (e.g. several 10x
# tiles) may still legitimately overlap each other -- that's fine, handled
# by assemble_mosaic_canvas's own stacking-order compositing.
MOSAIC_KEEP_OBJECTIVES = ["10x"]

tiles = tiles_all if MOSAIC_KEEP_OBJECTIVES is None else [
    t for t in tiles_all if t.objective_name in MOSAIC_KEEP_OBJECTIVES
]
n_excluded = len(tiles_all) - len(tiles)
print(f"Using {len(tiles)} tile(s)"
      + (f" (excluded {n_excluded} not in {MOSAIC_KEEP_OBJECTIVES})" if n_excluded else ""))

# Majority objective among the *kept* tiles -- trivially the one kept
# objective when MOSAIC_KEEP_OBJECTIVES has a single entry (the default);
# only matters if you widen MOSAIC_KEEP_OBJECTIVES to composite more than
# one objective, since the histogram cell below estimates THRESHOLD from
# just the majority one (see that cell's markdown for why).
kept_obj_counts = Counter(t.objective_name for t in tiles)
MAJORITY_OBJECTIVE = max(kept_obj_counts, key=kept_obj_counts.get)

WORKING_PIXEL_UM = 5.0   # canvas resolution -- smaller = sharper but slower/more memory
canvas = assemble_mosaic_canvas(tiles, working_pixel_um=WORKING_PIXEL_UM)
print(f"Canvas: {canvas.image.shape[1]}x{canvas.image.shape[0]} px "
      f"at {canvas.pixel_size_um:.2f} um/px, origin {canvas.origin_um}")

## Inspect intensity to pick a threshold

Estimated from just the majority-objective tiles (`MAJORITY_OBJECTIVE`), not
the full mixed-scale `tiles` set: a minority objective at a very different
exposure/gain (e.g. a handful of high-mag reference FOVs) can add a third
mode to the combined histogram and confuse simple two-peak valley detection,
even though the canvas itself still composites every tile spatially. Every
majority-objective tile's log-space histogram is shown (thin gray lines,
`alpha=0.5`), plus a solid black histogram pooling their pixels together (all
density-normalized, same bins, so they're directly comparable). When that
combined histogram is clearly bimodal, the valley between the background
peak and the tissue-signal peak is estimated automatically, drawn as a red
dashed line labelled with its value in linear intensity units, and used
below as the starting `THRESHOLD`.

In [ ]:
tiles_for_threshold = [t for t in tiles if t.objective_name == MAJORITY_OBJECTIVE]

ax_hist, ESTIMATED_THRESHOLD = plot_tile_intensity_histograms(tiles_for_threshold)
ax_hist.figure.set_size_inches(8, 5)
ax_hist.figure.tight_layout()

print(f"Estimated threshold from {len(tiles_for_threshold)} '{MAJORITY_OBJECTIVE}' tile(s) "
      f"(valley between background/tissue peaks): {ESTIMATED_THRESHOLD}")

## Segment tissue + holes -- tune and re-run

Start with the defaults; re-run this cell and the plot cell below with
adjusted parameters until the overlay looks right. `THRESHOLD` starts at
`ESTIMATED_THRESHOLD` (the valley read off the histogram above) rather than
`None`/Otsu, so the first iteration already uses the better estimate --
override with a fixed number, or set back to `None` for Otsu, if needed. All
distances are in real microns, not canvas pixels, so they carry over even if
you change `WORKING_PIXEL_UM` above.

- `SMOOTH_SIGMA_UM` -- Gaussian blur before thresholding, to suppress
  per-tile illumination speckle. Too small: noisy/fragmented mask. Too
  large: blurs away real fine structure.
- `CLOSE_RADIUS_UM` -- bridges small real gaps between adjacent bits of the
  same tissue piece so they merge into one polygon instead of many.
- `OPEN_RADIUS_UM` -- removes small noise specks that survive closing.
- `MARGIN_UM` -- outward safety buffer applied after cleanup (a hand-drawn
  boundary naturally includes some margin beyond the exact signal edge).
- `MIN_TISSUE_AREA_UM2`/`MIN_HOLE_AREA_UM2` -- drop components smaller than
  this after morphology.
- `MIN_ISLAND_AREA_UM2` -- a hole can itself enclose a real tissue island (a
  true donut/annulus -- e.g. a ring of tissue with more tissue in the
  middle); islands at least this big become interior rings of the hole
  polygon (dashed in the review plot below) so they're correctly excluded
  *from* the hole rather than swallowed whole into a solid disk.
- `SIMPLIFY_TOL_UM` -- polygon simplification tolerance (marching squares
  otherwise emits one vertex per canvas pixel of perimeter).

In [ ]:
THRESHOLD           = ESTIMATED_THRESHOLD   # from the histogram cell above; None = Otsu instead
SMOOTH_SIGMA_UM     = 10.0
CLOSE_RADIUS_UM     = 50.0
OPEN_RADIUS_UM      = 15.0
MARGIN_UM           = 75.0
MIN_TISSUE_AREA_UM2 = 1000.0
MIN_HOLE_AREA_UM2   = 500.0
MIN_ISLAND_AREA_UM2 = 1000.0
SIMPLIFY_TOL_UM     = 15.0

segmentation = segment_mosaic_tissue(
    canvas,
    threshold           = THRESHOLD,
    smooth_sigma_um     = SMOOTH_SIGMA_UM,
    close_radius_um     = CLOSE_RADIUS_UM,
    open_radius_um      = OPEN_RADIUS_UM,
    margin_um           = MARGIN_UM,
    min_tissue_area_um2 = MIN_TISSUE_AREA_UM2,
    min_hole_area_um2   = MIN_HOLE_AREA_UM2,
    min_island_area_um2 = MIN_ISLAND_AREA_UM2,
    simplify_tol_um     = SIMPLIFY_TOL_UM,
)
print(f"Threshold used : {segmentation.threshold:.1f}")
print(f"Tissue pieces  : {len(segmentation.tissue_polygons)}")
print(f"Holes          : {len(segmentation.hole_polygons)}")
n_islands = sum(len(p.interiors) for p in segmentation.hole_polygons)
print(f"Islands inside holes: {n_islands}")
for i, p in enumerate(sorted(segmentation.tissue_polygons, key=lambda p: -p.area)):
    print(f"  tissue[{i}] area={p.area:,.0f} um^2")

In [ ]:
ax = plot_mosaic_segmentation(canvas, segmentation)
ax.figure.set_size_inches(10, 10)
ax.figure.tight_layout()

## Write boundary_positions*.txt / hole*.txt

Only run this once the plot above looks right. Writes to
`positions/boundaries/from_mosaic/`. A single detected tissue piece is
written as the legacy `boundary_positions.txt`; several disjoint pieces are
written as `boundary_positions_{b}.txt` (the "single" layout -- see
`discover_boundary_files`). Existing files with the same name are
overwritten, so re-running after a parameter change is safe.

After this, continue with `02_create_positions_from_boundaries.ipynb`.

In [ ]:
written = save_boundary_from_mosaic(segmentation, OUTPUT_DIR)
print(f"Wrote {len(written)} file(s) to {OUTPUT_DIR}:")
for f in written:
    print(f"  {f}")